# 🎬 VoxStudio Server — Google Colab

Deploy VoxStudio (OmniVoice TTS + Whisper + Edge TTS dubbing) trên Colab GPU.

**Chạy tuần tự các cell 1 → 6** để khởi động server, rồi dùng cell test để dub video.


In [ ]:
#@title 1️⃣ Kiểm tra GPU
import torch, subprocess
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    try: print(subprocess.check_output(["nvidia-smi", "--query-gpu=memory.total,memory.free", "--format=csv"]).decode())
    except: pass
else:
    print("⚠️ No GPU — vào Runtime → Change runtime type → GPU")


In [ ]:
#@title 2️⃣ Clone/pull code từ GitHub
import os, shutil

REPO = "https://github.com/thuantiensd/VoxStudio.git"

if os.path.exists("/content/VoxStudio/.git"):
    print("📥 Pulling latest code...")
    !cd /content/VoxStudio && git pull
else:
    print("📥 Cloning repo...")
    !git clone {REPO} /content/VoxStudio

# Copy server + OmniVoice source
for src, dst in [("/content/VoxStudio/server", "/content/server"),
                 ("/content/VoxStudio/OmniVoice-master", "/content/OmniVoice-master")]:
    if os.path.exists(dst):
        shutil.rmtree(dst, ignore_errors=True)
    shutil.copytree(src, dst)

print("\n=== /content/server ===")
!ls /content/server/
print("\n✅ Code ready!")


In [ ]:
#@title 3️⃣ Cài đặt dependencies
import subprocess, sys, os
os.chdir("/content")

def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd="/content")
    if r.returncode != 0:
        err = (r.stderr or r.stdout or "").strip().split("\n")[-3:]
        print(f"  ⚠️ {' | '.join(err)}")
    return r.returncode == 0

print("📦 Installing core packages...")
run("pip install -q fastapi 'uvicorn[standard]' python-multipart soundfile pyngrok")

print("📦 Installing translation...")
run("pip install -q deep-translator google-generativeai")

print("📦 Installing AI models...")
run("pip install -q transformers accelerate bitsandbytes")

print("📦 Installing Faster-Whisper + scipy...")
run("pip install -q faster-whisper scipy")

print("📦 Installing OmniVoice TTS from source...")
!pip install -e /content/OmniVoice-master 2>&1 | tail -3

print("📦 Installing Demucs...")
run("pip install -q demucs")

print("📦 Installing ffmpeg...")
run("apt-get install -y -qq ffmpeg > /dev/null 2>&1")

print("📦 Installing remaining from requirements.txt...")
run("pip install -q -r /content/server/requirements.txt")

# Verify (clear stale cached imports first)
print("\n=== Verify ===")
try:
    import importlib, sys
    for m in ["omnivoice"]:
        if m in sys.modules: del sys.modules[m]
    import omnivoice
    print(f"✅ omnivoice {omnivoice.__version__}")
except Exception as e:
    print(f"❌ omnivoice: {e} — thử Runtime → Restart Runtime rồi chạy lại từ cell 3")

try:
    import torch
    print(f"✅ torch {torch.__version__} (CUDA: {torch.cuda.is_available()})")
except: pass

try:
    from faster_whisper import WhisperModel
    print("✅ faster-whisper")
except Exception as e:
    print(f"❌ faster-whisper: {e}")

try:
    import edge_tts
    print("✅ edge-tts")
except Exception as e:
    print(f"❌ edge-tts: {e}")

print("\n✅ Done!")


In [ ]:
#@title 4️⃣ Setup ngrok (optional — truy cập server từ ngoài)
NGROK_TOKEN = ""  #@param {type:"string"}

if NGROK_TOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    print(f"✅ ngrok token configured")
else:
    print("⚠️ Không có ngrok token — chỉ dùng được localhost trong notebook (đủ để test)")


In [ ]:
#@title 5️⃣ Cấu hình
import os

# Default config for Colab CUDA (cao chất lượng)
os.environ["DEVICE"] = "cuda"
os.environ["USE_FASTER_WHISPER"] = "true"
os.environ["FASTER_WHISPER_MODEL"] = "large-v3-turbo"

# LLM for Qwen-based translation polish (only CUDA, >15GB VRAM)
# Uncomment to enable:
# os.environ["LLM_MODEL"] = "Qwen/Qwen2.5-7B-Instruct"

# Gemini API (optional context-aware translation)
GEMINI_API_KEY = ""  #@param {type:"string"}
if GEMINI_API_KEY:
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

print("✅ Config set")
for k in ["DEVICE", "USE_FASTER_WHISPER", "FASTER_WHISPER_MODEL"]:
    print(f"  {k} = {os.environ.get(k)}")


In [ ]:
#@title 6️⃣ 🚀 Khởi động Server
import os, subprocess, time, signal

os.chdir("/content")

# Kill old server if any
try:
    old = subprocess.check_output(["pgrep", "-f", "uvicorn app.main"], text=True).strip().split()
    for pid in old:
        os.kill(int(pid), signal.SIGTERM)
    time.sleep(2)
    print("🛑 Killed old server(s)")
except: pass

# Start new server
env = os.environ.copy()
env["PYTHONPATH"] = "/content/server"

proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/server",
    env=env,
    stdout=open("/content/server.log", "w"),
    stderr=subprocess.STDOUT,
)

print(f"🚀 Server PID: {proc.pid}")
print("⏳ Loading Whisper (~20-40s on CUDA)...")

import requests
ok = False
for i in range(60):
    try:
        r = requests.get("http://localhost:8000/health", timeout=2)
        if r.status_code == 200:
            ok = True
            print(f"✅ Server ready: {r.json()}")
            break
    except: pass
    time.sleep(2)

if not ok:
    print("❌ Server failed to start — xem log cell dưới")


In [ ]:
#@title 📊 Xem server logs
!tail -80 /content/server.log


In [ ]:
#@title 📊 Check VRAM + loaded models
import requests
r = requests.get("http://localhost:8000/system/vram")
print(r.json())


In [ ]:
#@title 🧪 Health check
import requests
print(requests.get("http://localhost:8000/health").json())
print(requests.get("http://localhost:8000/api/v1/dubbing/projects").json())


## 🎬 Test Full Dubbing Pipeline

Cell dưới — upload video, tự động dub bằng OmniVoice (BLV voice mặc định) + Google Translate.


In [ ]:
#@title 🎬 Full Dubbing (OmniVoice BLV mặc định)
import requests, json
from google.colab import files

BASE = "http://localhost:8000/api/v1"

# 1. Upload video
print("📤 Upload video để dub...")
up = files.upload()
vname = list(up.keys())[0]
with open(f"/content/{vname}", "wb") as f: f.write(up[vname])

# 2. Create project
with open(f"/content/{vname}", "rb") as vf:
    r = requests.post(f"{BASE}/dubbing/projects",
        files={"video": (vname, vf, "video/mp4")},
        data={"target_language": "vietnamese",
              "source_language": "auto",
              "enable_dubbing": "true"})
pid = r.json()["id"]
print(f"✅ project={pid}")

# 3. Auto-dub
print("\n🚀 Auto-dub (Demucs → Whisper → Translate → TTS → Export)...\n")
r = requests.post(f"{BASE}/dubbing/projects/{pid}/auto-dub", stream=True)
for line in r.iter_lines(decode_unicode=True):
    if line and line.startswith("data: "):
        try:
            d = json.loads(line[6:])
            pct = d.get("progress", 0)
            bar = "█" * int(pct/5) + "░" * (20-int(pct/5))
            print(f"  [{bar}] {pct}% — {d.get('label','')}")
            if d.get("step") in ("error", "done"): break
        except: pass

# 4. Download
dl = requests.get(f"{BASE}/dubbing/projects/{pid}/export/download")
if dl.status_code == 200:
    out = f"/content/dubbed_{vname}"
    with open(out, "wb") as f: f.write(dl.content)
    print(f"\n✅ {out} ({len(dl.content)/1024/1024:.1f}MB)")
    files.download(out)
else:
    print(f"❌ download failed: {dl.status_code}")


## 🎙️ Dubbing với giọng clone (My Voice)

In [ ]:
#@title 🎙️ Upload voice reference + clone + dub
import requests, json
from google.colab import files

BASE = "http://localhost:8000/api/v1"

# 1. Upload reference voice WAV (3-10s)
print("📤 Upload reference voice WAV (giọng muốn clone)...")
up = files.upload()
ref_name = list(up.keys())[0]
with open(f"/content/{ref_name}", "wb") as f: f.write(up[ref_name])

# 2. Clone voice
with open(f"/content/{ref_name}", "rb") as rf:
    r = requests.post(f"{BASE}/voices/clone",
        files={"audio": (ref_name, rf, "audio/wav")},
        data={"name": "CloneTest"})
if r.status_code != 200:
    print(f"❌ Clone failed: {r.text[:300]}")
    raise SystemExit
voice_id = r.json()["id"]
print(f"✅ voice_id={voice_id}")
print(f"   ref_text={r.json().get('ref_text', '')[:80]}...")

# 3. Upload target video
print("\n📤 Upload video cần dub...")
up2 = files.upload()
vname = list(up2.keys())[0]
with open(f"/content/{vname}", "wb") as f: f.write(up2[vname])

# 4. Create project with voice_id
with open(f"/content/{vname}", "rb") as vf:
    r = requests.post(f"{BASE}/dubbing/projects",
        files={"video": (vname, vf, "video/mp4")},
        data={"target_language": "vietnamese",
              "source_language": "auto",
              "enable_dubbing": "true",
              "voice_id": voice_id})
pid = r.json()["id"]
print(f"✅ project={pid}")

# 5. Auto-dub
print("\n🚀 Auto-dub với giọng clone...\n")
r = requests.post(f"{BASE}/dubbing/projects/{pid}/auto-dub", stream=True)
for line in r.iter_lines(decode_unicode=True):
    if line and line.startswith("data: "):
        try:
            d = json.loads(line[6:])
            pct = d.get("progress", 0)
            bar = "█" * int(pct/5) + "░" * (20-int(pct/5))
            print(f"  [{bar}] {pct}% — {d.get('label','')}")
            if d.get("step") in ("error", "done"): break
        except: pass

# 6. Download
dl = requests.get(f"{BASE}/dubbing/projects/{pid}/export/download")
if dl.status_code == 200:
    out = f"/content/clone_dubbed_{vname}"
    with open(out, "wb") as f: f.write(dl.content)
    print(f"\n✅ {out} ({len(dl.content)/1024/1024:.1f}MB)")
    files.download(out)
else:
    print(f"❌ download: {dl.status_code}")


In [ ]:
#@title ⏹️ Dừng server
!pkill -f "uvicorn app.main" 2>/dev/null
print("🛑 Server stopped")
